# Imports

In [ ]:
import pandas
from minio import Minio
import json
import pendulum

# Initialize Minio Client

In [ ]:
minio_client = Minio(
    "localhost:9000",
    access_key="root",
    secret_key="password",
    secure=False
)

# Get all objects from bucket

In [ ]:
bucket_name = "bga-logs-server-test4"
objects = list(minio_client.list_objects(bucket_name))

In [ ]:
import copy

def deep_merge(dict1, dict2):
    result = copy.deepcopy(dict1)
    
    for key, value in dict2.items():
        if key in result and isinstance(result[key], dict) and isinstance(value, dict):
            result[key] = deep_merge(result[key], value)
        elif key in result and isinstance(result[key], list) and isinstance(value, list):
            result[key] = merge_lists(result[key], value)
        else:
            result[key] = copy.deepcopy(value)
    
    return result

def merge_lists(list1, list2):
    merged = copy.deepcopy(list1)
    
    for item in list2:
        if isinstance(item, dict):
            found = next((i for i, x in enumerate(merged) if isinstance(x, dict)), None)
            if found is not None:
                merged[found] = deep_merge(merged[found], item)
            else:
                merged.append(copy.deepcopy(item))
        else:
            merged.append(copy.deepcopy(item))
    
    return merged

In [ ]:
payloads = dict({})
for obj in sorted(objects, key=lambda x: x.object_name):
    my_data = json.loads(minio_client.get_object(bucket_name, obj.object_name).read())
    for payload_item in my_data["payload"]:
            payloads.update(**payload_item)

In [ ]:
needed_types = dict({})
wanted_type = ["giveCards", "newHand", "takeCards", "tableWindow", "playCard", "giveAllCardsToPlayer", "gameStateChange", "earlyEnd", "tableInfosChanged"]
for obj in sorted(objects, key=lambda x: x.object_name):
    my_data = json.loads(minio_client.get_object(bucket_name, obj.object_name).read())
    for payload_item in my_data["payload"]:
        channel = payload_item["channel"]
        if channel not in needed_types:
            needed_types[channel] = {}
        for data_item in payload_item["data"]:
            data_type = data_item["type"]
            if data_type not in wanted_type:
                continue
            if data_type in needed_types[channel]:
                deep_merge(needed_types[channel][data_type], data_item)
            else:
                needed_types[channel][data_type] = data_item

In [ ]:
import pyperclip
channel_data = needed_types["/table/t568341597"]["tableInfosChanged"]
pyperclip.copy(json.dumps(channel_data))
print(channel_data)

In [ ]:
def remove_duplicate_columns(df):
    columns = df.columns
    duplicated_cols = columns[columns.duplicated()].unique()
    if len(duplicated_cols) > 0:
        print(duplicated_cols)
    for col in duplicated_cols:
        cols = df.loc[:, df.columns == col]
        combined = cols.bfill(axis=1).iloc[:, 0]
        df = df.drop(columns=col).assign(**{col: combined})
    aligned_df = df.reindex(columns=df.columns)
    return aligned_df

In [ ]:
from pendulum.parsing.exceptions import ParserError
dataframes = []
for obj in sorted(objects, key=lambda x: x.object_name):
    response = minio_client.get_object(bucket_name, obj.object_name)
    content = response.read()
    json_data: dict = json.loads(content)
    if len(json_data) == 0 or json_data == {"message": "2"}:
        minio_client.remove_object(bucket_name, obj.object_name)
    file_name = obj.object_name.replace(".json", "")
    try:
        file_time: pendulum.DateTime = pendulum.parse(file_name)
    except ParserError:
        file_split = int(file_name.split("-")[0])
        file_time = pendulum.from_timestamp(file_split)
    json_data["file_time"] = file_time
    if json_data.get("payload", None) is not None:
        payload = json_data.pop("payload")
        for payload_item in payload:
            metadata = {k: v for k,v in json_data.items()}
            metadata.update({k: v for k,v in payload_item.items() if k != "data"})
            if "data" in payload_item:
                for item in payload_item["data"]:
                    if "log" in item and not item["log"]: del item["log"]
                    if "time" in item: del item["time"]
                    normalized = pandas.json_normalize(item)
                    meta_df = pandas.DataFrame([metadata])
                    meta_df.reindex(columns=meta_df.columns.append(normalized.columns))
                    aligned_df = pandas.concat([meta_df, normalized], axis=1)
                    aligned_df = remove_duplicate_columns(aligned_df)
                    aligned_df = aligned_df.reindex(columns=meta_df.columns.append(normalized.columns))
                    dataframes.append(aligned_df)
            else:
                meta_df = pandas.DataFrame([metadata])
                aligned_df = meta_df.reindex(columns=meta_df.columns)
                dataframes.append(meta_df)
    else:
        df = pandas.DataFrame([json_data])
        aligned_df = df.reindex(columns=df.columns)
        dataframes.append(df)

In [ ]:
combined_df = pandas.concat(dataframes).reset_index(drop=True)
combined_df.sort_values(by="file_time", ascending=True).head(100)

In [ ]:
data_type_groups = combined_df.groupby(["channel", "type"])

In [ ]:
for group, _ in data_type_groups:
    channel, type_value = group
    print("Channel: " + channel + " Type: " + type_value)
    

In [ ]:
type_groups = combined_df.groupby(["channel"])
combined_df.loc[type_groups["file_time"].idxmin()][["file_time", "type", "channel"]].sort_values(["file_time", "channel"])

In [ ]:
from rich.pretty import pprint
columns_list = sorted(list(data_type_groups.get_group(("/player/p89368681", "shouldAcceptGameStart")).dropna(how="all").dropna(axis=1, how="all").columns))
column_len = len(columns_list)
pprint((columns_list, column_len))#798CB9

In [ ]:
data_type_groups.get_group(("/player/p93502223", "shouldAcceptGameStart")).dropna(axis=1, how="all").head()

In [ ]:
card_groups = data_type_groups.get_group(("/player/p93502223", "playCard")).dropna(axis=1, how="all")[["args.card.id", "args.card.type", "args.card.type_arg", "args.color_displayed", "args.value_displayed"]].groupby(["args.card.type", "args.value_displayed"])
for thing, _ in card_groups:
    print(card_groups.get_group(thing)[["args.card.id", "args.value_displayed", "args.card.type"]].head(1))

#### Channel Analysis

In [ ]:
combined_df["channel"].unique()

In [ ]:
channel_groups = combined_df.groupby("channel")

In [ ]:
for type_value, _ in channel_groups:
    print("Channel type: " + type_value)

#### Data frames for each channel

In [ ]:
player_df = combined_df[combined_df["channel"].str.startswith("/player/")]
table_df = combined_df[combined_df["channel"].str.startswith("/table/")]
table_manager_df = combined_df[combined_df["channel"].str.startswith("/tablemanager/")]

## Data Exploration with the table messages

In [ ]:
table_df.sort_values(by="file_time", ascending=True).head(50)

In [ ]:
table_groups = table_df.groupby("type")

In [ ]:
for type_value, group_df in table_groups:
    print("Type: " +type_value)

In [ ]:
tail_df: pandas.DataFrame = table_groups.get_group("").sort_values(by="file_time", ascending=True)
tail_df.dropna(axis=1, how="all").head(50)


In [ ]:
player_columns = [col for col in tail_df.columns if col.startswith('args.players.')]
tail_df[player_columns].head(50)

## Data Exploration with the player messages

In [ ]:
player_df["channel"].unique()

In [ ]:
player_df.sort_values(by="file_time", ascending=True).dropna(how="all").dropna(axis=1, how="all").head(50)

In [ ]:
player_groups = player_df.groupby("type")

In [ ]:
for type_value, group_df in player_groups:
    print("Type: " +type_value)

In [ ]:

player_tail: pandas.DataFrame = player_groups.get_group("").sort_values(by="file_time", ascending=True)
player_tail.dropna(axis=1, how="all").head(50)